# CEDAR Query API - Example Search Scenarios and Queries

The Cancer Epitope Database and Analysis Resource (CEDAR) API (Application Programming Interface) enables users to perform queries and retrieve CEDAR data programmatically within their preferred environment. This Python Notebook Series illustrates basic usage examples of the CEDAR API using Python.

The CEDAR API is built on a PostgREST platform, providing transparent access to the tables on the backend. For more information on the expressive syntax of PostgREST, refer to this [document](https://postgrest.org/en/stable/api.html#). The CEDAR data can be queried through search and export endpoints described in the [CEDAR API documentation](https://discuss.iedb.org/t/application-programming-interface-api/220) and the interactive [Swagger documentation](https://cedar-api.iedb.org/docs/swagger/).

These example search scenarios and queries are derived from this [Book Chapter](https://doi.org/10.1007/978-1-0716-4566-6_3), which describes how to perform different queries to the [CEDAR homepage](https://cedar.iedb.org). This notebook reproduces the same queries using the CEDAR API.

## Research scenario III

Certain viruses can cause cancer by integrating viral DNA into the human genome. As viral antigens are foreign to the human immune system, they are attractive targets for immunotherapy. To explore putative antigen and epitope targets, the user wants to retrieve a list of viral antigens and epitopes in head and neck cancer with positive outcomes in T cell recognition assays.

### Approach

To retrieve this data, we will execute the following steps:

1. **Search T cell assays** - Identify positive T cell assays of viral-derived epitopes in the context of head and neck cancer
2. **Map associated assay information** - Retrieve complete assay information

Each step requires a request to a different interconnected endpoint. For more information, refer to the [PostgREST resource embedding documentation](https://docs.postgrest.org/en/v12/references/api/resource_embedding.html#resource-embedding).

### Setup

First, import the required modules and define a function to make CEDAR API requests. This function takes the endpoint, the search parameters, and the base URI (Unified Resource Identifier) as inputs. Based on these, it constructs the URL (Unified Resource Locator) to interact with the data and resources specified. Then, it performs the API request. Since the API has a maximum limit of 10,000 rows per request, the function iterates through results, retrieving data in chunks. Finally, it parses the response and returns a pandas DataFrame. For more information, refer to [pandas documentation](https://pandas.pydata.org/).

In [1]:
import os
import requests
import pandas as pd
import io
import time

pd.set_option('display.max_columns', 100)

def API_query(endpoint, query_params, base_uri='https://cedar-api.iedb.org/'):
    """
    Execute a query against the CEDAR API with pagination support.

    Parameters:
    -----------
    endpoint : str
        The API endpoint to query
    query_params : dict
        Dictionary of query parameters
    base_uri : str
        Base URI for the CEDAR API

    Returns:
    --------
    pd.DataFrame
        Combined results from all paginated requests
    """

    url = os.path.join(base_uri, endpoint)
    df = pd.DataFrame()

    # set the offset to 0
    query_params['offset'] = 0

    # loop through the pages of results
    # API only allows pulling 10,000 entries at a time
    while(True):
        print('Fetching offset: %i' % query_params['offset'])
        r = requests.get(
            url,
            params=query_params,
            headers={'accept': 'text/csv', 'Prefer': 'count=exact'}
        )

        try:
            # Parse CSV response and append to existing DataFrame
            df = pd.concat([df, pd.read_csv(io.StringIO(r.content.decode('utf-8')))])
            query_params['offset'] += 10000
        except pd.errors.EmptyDataError:
            # No more data available
            break

        # Rate limiting: pause between requests to not overload the server
        time.sleep(1)

    return df


### Step 1: Search T cell assays

In this example, we will search for **positive T cell assays** of **viral epitopes** in the context of **head and neck cancer**.

The CEDAR API provides two types of endpoints, the search and export. The search endpoints contain fields with information that facilitate to programatically identify and/or filter the data of interest, while the export endpoints organize the data in a user-friendly format, matching the structure and naming conventions of the custom exports on the CEDAR website.

To perform our query, we will first use the `tcell_search` endpoint. Let's examine its structure and available fields.

---



In [2]:
cedar_url='https://cedar-api.iedb.org/'
endpoint='tcell_search'

table = pd.json_normalize(requests.get(os.path.join(cedar_url, endpoint + '?limit=3')).json())
display(table)

,tcell_id,tcell_iri,structure_id,structure_iri,linear_sequence,structure_type,structure_description,curated_source_antigen,reference_id,reference_iri,reference_type,pubmed_id,reference_authors,reference_titles,reference_dates,journal_name,epitope_summary,reference_summary,immunization_description,antigen_description,antigen_er,mhc_restriction,assay_description,e_modification,linear_sequence_length,qualitative_measure,mhc_class,mhc_allele_resolution,submission_id,submission_iri,pdb_id,chebi_ids,mhc_allele_evidence,antibody_isotype,direct_ex_vivo_bool,receptor_ids,receptor_group_ids,tcr_receptor_group_ids,bcr_receptor_group_ids,receptor_group_iris,tcr_receptor_group_iris,bcr_receptor_group_iris,receptor_types,receptor_names,receptor_chain1_types,receptor_chain2_types,receptor_chain1_full_seqs,receptor_chain2_full_seqs,receptor_chain1_cdr1_seqs,receptor_chain2_cdr1_seqs,receptor_chain1_cdr2_seqs,receptor_chain2_cdr2_seqs,receptor_chain1_cdr3_seqs,receptor_chain2_cdr3_seqs,host_organism_iri_search,host_organism_iri,host_organism_name,source_organism_iri_search,source_organism_iri,source_organism_name,mhc_allele_iri_search,mhc_allele_iri,mhc_allele_name,parent_source_antigen_iri_search,parent_source_antigen_iri,parent_source_antigen_name,parent_source_antigen_source_org_iri,parent_source_antigen_source_org_name,disease_iri_search,disease_iris,disease_names,assay_iri_search,assay_iris,assay_names,epitope_structure_defined,non_peptidic_molecule_iri_search,non_peptidic_molecule_iri,non_peptidic_molecule_name,r_object_source_molecule_iri_search,r_object_source_molecule_iri,r_object_source_molecule_name,r_object_source_organism_iri_search,r_object_source_organism_iri,r_object_source_organism_name,e_related_object_type,host_mhc_types_present,neoantigen_bool,viral_antigen_bool,germline_antigen_bool,other_antigen_bool,disease_stages,naturally_occuring_disease_bool,animal_model_of_cancer_bool,all_vaccination_bool,prophylactic_vaccination_bool,therapeutic_vaccination_bool,mutation
0,25872362,CEDAR_ASSAY:25872362,1312162,CEDAR_EPITOPE:1312162,ALNSEALSVV,Linear peptide,ALNSEALSVV,None,1047949,CEDAR_REFERENCE:1047949,Literature,40872950,[Lena Pfitzer; Gitta Boons; Lien Lybaert; Wim ...,[Accelerating Neoantigen Discovery: A High-Thr...,[2025],Vaccines (Basel),ALNSEALSVV,Lena Pfitzer;<br/>Vaccines (Basel)<br/>2025,Primary induction in vitro with ALNSEALSVV (Ep...,ALNSEALSVV,Epitope,None,ELISPOT<br/>IFNg release<br/><strong>Negative<...,None,10,Negative,None,None,None,None,None,None,None,None,0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"[NCBITaxon:314295, NCBITaxon:40674, NCBITaxon:...",NCBITaxon:9606,Homo sapiens (human),None,None,None,None,None,None,None,None,None,None,None,"[ONTIE:0003423, ONTIE:0003543]",[ONTIE:0003423],[healthy],"[OBI:0001414, OBI:0002055, OBI:0003128, OBI:11...",OBI:1110059,IFNg release|ELISPOT,Exact Epitope,None,None,None,None,None,None,"[NCBITaxon:1, NCBITaxon:2759, NCBITaxon:314295...",NCBITaxon:9606,Homo sapiens (human),fusion neo-epitope,None,1,0,0,0,None,1,0,0,0,0,None
1,2103810,CEDAR_ASSAY:2103810,2192,CEDAR_EPITOPE:2192,AKFVAAWTLKAAA,Linear peptide,AKFVAAWTLKAAA,None,1027564,CEDAR_REFERENCE:1027564,Literature,24560029,[Hayk Davtyan; Anahit Ghochikyan; Irina Petrus...,[The MultiTEP platform-based Alzheimer's disea...,[2014],Alzheimers Dement,AKFVAAWTLKAAA,Hayk Davtyan;<br/>Alzheimers Dement<br/>2014,Prophylactic vaccination with AKFVAAWTLKAAA (E...,AKFVAAWTLKAAA,Epitope,H2 class II,ELISPOT<br/>IFNg release<br/><strong>Positive<...,None,13,Positive,II,class,None,None,None,None,Cited reference,None,1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"[NCBITaxon:40674, NCBITaxon:9443, NCBITaxon:95...",NCBITaxon:9544,Macaca mulatta (rhesus macaque),None,None,None,"[GO:0032991, GO:0042611, MRO:0001356, MRO:0001...",MRO:0001461,mouse,None,None,None,None,None,None,None,None,"[OBI:0001414, OBI:0002055, OBI:0003128, OBI:11...",OBI

Next, we define the search parameters, which consist of three components:

- The **query filters** that we want to impose on the data.

  To filter viral antigens, we will use the `viral_antigen_bool` field, whose value is 1 if the antigen is from a viral source, and 0 otherwise. Alternatively, we could use the field `source_organism_iri_search` and filter by a taxon of interest, such as [Viruses](https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?mode=Info&id=10239&lvl=3&lin=f&keep=1&srchmode=1&unlock).

  To filter by head and neck cancer disease, we will use the field `disease_iri_search` which contains a tree of IRIs (Internationalized Resource Identifiers) that point to external resources such as [The Disease Ontology](https://disease-ontology.org/), allowing users to filter the data using stable identifiers. The Disease Ontology Identifier (DOID) of Head and Neck Cancer is [11934](https://disease-ontology.org/?id=DOID:11934).

  Also, we want to retrieve only the positive assays, so use the `qualitative_measure` field to only retrieve positive assays.

  To apply these filters, we utilize operators such as `eq`, `cs`, and `in`, which allow us to perform flexible queries and refine the search. For more information on the operators' syntax, refer to this [documentation](https://docs.postgrest.org/en/v12/references/api/tables_views.html#operators).

- The **pagination parameters** required to retrieve the results in chunks. The order parameter is a field used to sort and paginate the data so that each retrieved chunk contains different consecutive rows. The offset parameter indicates the index of the first element of each data chunk.

- The final **column selection**. In this case, we will keep the assay IDs of the `tcell_id` field. These will be used for subsequent queries to other endpoints.

In [3]:
search_params={
    # Query filters
    'viral_antigen_bool': 'eq.1',
    'disease_iri_search':'cs.{DOID:11934}',
    'qualitative_measure':'in.(Positive,Positive-Low,Positive-Intermediate,Positive-High)',

    # Pagination
    'order': 'tcell_id',
    'offset': 0,

    # Column selection
    'select':'tcell_id',
}

# Execute epitope search
tcell_df = API_query(endpoint, search_params)

print(f"Retrieved {len(tcell_df)} epitope records")
tcell_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 132 epitope records


,tcell_id
0,5566220
1,5566222
2,5566223
3,5566224
4,5566225


### Step 2: Map associated T cell assays

With the T cell assay IDs of interest, we map the complete information of the assays by querying the ``tcell_export`` endpoint. This requires formatting the `tcell_ids`, and including them in the search parameters.

In [4]:
tcell_ids = ','.join(map(str,set(tcell_df['tcell_id'].to_list())))

endpoint='tcell_export'

search_params = {
    # Query filter
    'assay_id': 'in.(' + tcell_ids + ')',

    # Pagination
    'order': 'assay_id',
    'offset': 0,
}

# Execute final query
tcell_assays_df = API_query(endpoint, search_params)

print(f"Retrieved {len(tcell_assays_df)} T cell assays")
tcell_assays_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 132 T cell assays


,assay_id,assay_id__cedar_iri,reference__cedar_iri,reference__type,reference__pmid,reference__submission_id,reference__authors,reference__journal,reference__date,reference__title,epitope__cedar_iri,epitope__object_type,epitope__name,epitope__reference_name,epitope__modified_residues,epitope__modifications,epitope__starting_position,epitope__ending_position,epitope__iri,epitope__synonyms,epitope__source_molecule,epitope__source_molecule_iri,epitope__molecule_parent,epitope__molecule_parent_iri,epitope__source_organism,epitope__source_organism_iri,epitope__species,epitope__species_iri,epitope__mutation,epitope__epitope_comments,related_object__epitope_relation,related_object__object_type,related_object__name,related_object__starting_position,related_object__ending_position,related_object__iri,related_object__synonyms,related_object__source_molecule,related_object__source_molecule_iri,related_object__molecule_parent,related_object__molecule_parent_iri,related_object__source_organism,related_object__source_organism_iri,related_object__species,related_object__species_iri,host__name,host__iri,host__geolocation,host__geolocation_iri,host__sex,...,in_vitro_immunogen__source_organism_iri,in_vitro_immunogen__species,in_vitro_immunogen__species_iri,adoptive_transfer__flag,adoptive_transfer__comments,immunization__comments,assay__location_of_assay_data_in_reference,assay__method,assay__response_measured,assay__units,assay__iri,assay__qualitative_measurement,assay__measurement_inequality,assay__quantitative_measurement,assay__number_of_subjects_tested,assay__number_of_subjects_positive,assay__response_frequency_,assay__comments,effector_cell__source_tissue,effector_cell__source_tissue_iri,effector_cell__name,effector_cell__iri,effector_cell__culture_condition,effector_cell__tcr_name,complex__pdb_id,antigen_presenting_cell__source_tissue,antigen_presenting_cell__source_tissue_iri,antigen_presenting_cell__name,antigen_presenting_cell__iri,antigen_presenting_cell__culture_condition,mhc_restriction__name,mhc_restriction__iri,mhc_restriction__evidence_code,mhc_restriction__evidence_iri,mhc_restriction__class,assay_antigen__epitope_relation,assay_antigen__object_type,assay_antigen__name,assay_antigen__reference_name,assay_antigen__starting_position,assay_antigen__ending_position,assay_antigen__iri,assay_antigen__source_molecule,assay_antigen__source_molecule_iri,assay_antigen__molecule_parent,assay_antigen__molecule_parent_iri,assay_antigen__source_organism,assay_antigen__source_organism_iri,assay_antigen__species,assay_antigen__species_iri
0,5566220,https://cedar.iedb.org/assay/5566220,https://cedar.iedb.org/reference/1033905,Literature,30154146,NaN,Sri Krishna; Peaches Ulrich; Eric Wilson; Falg...,Cancer Res,2018,Human Papilloma Virus Specific Immunogenicity ...,https://cedar.iedb.org/epitope/911857,Linear peptide,SPEIIRQHL,E2(207-215),NaN,NaN,207,215,NaN,NaN,E2,http://www.ncbi.nlm.nih.gov/protein/AAD33255.1,Regulatory protein E2,http://www.uniprot.org/uniprot/P03120,Human papillomavirus 16,http://purl.obolibrary.org/obo/NCBITaxon_333760,Alphapapillomavirus 9,http://purl.obolibrary.org/obo/NCBITaxon_337041,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Homo sapiens (human),http://purl.obolibrary.org/obo/NCBITaxon_9606,NaN,NaN,NaN,...,NaN,NaN,NaN,N,NaN,Blood sampels were collected from 22 patients ...,Figure 2,ELISPOT,IFNg release,NaN,http://purl.obolibrary.org/obo/OBI_1110059,Positive,NaN,NaN,22.0,2.0,9.0,NaN,blood,http://purl.obolibrary.org/obo/UBERON_0000178,PBMC,http://purl.obolibrary.org/obo/CL_2000001,Direct Ex Vivo,NaN,NaN,blood,http://purl.obolibrary.org/obo/UBERON_0000178,PBMC,http://purl.obolibrary.org/obo/CL_2000001,Direct Ex Vivo,HLA-B*07:02,http://purl.obolibrary.org/obo/MRO_0001062,MHC binding prediction,http://purl.obolibrary.org/obo/ECO_0008081,I,Epitope,Linear peptide,SPEIIRQHL,NaN,207.0,215.0,NaN,E2,http://www.ncbi.nlm.nih.gov/protein/AAD33255.1,Regulatory protein E2,http://www.uniprot.org/uniprot/P03120,H

### Results

The final DataFrame contains all the positive T cell assays of viral-derived epitopes in the context of head and neck cancer. This programmatic approach demonstrates the CEDAR API's capability for automated data retrieval and analysis.

## References

Koşaloğlu-Yalçın, Z. et al. (2025) *Using the Cancer Epitope Database and Analysis Resource (CEDAR)*. Alexander Krasnitz and Pascal Belleau (Eds.), *Cancer Bioinformatics, Methods in Molecular Biology*, vol. 2932. [https://doi.org/10.1007/978-1-0716-4566-6_3 ](https://doi.org/10.1007/978-1-0716-4566-6_3)

Koşaloğlu-Yalçın, Z. et al. (2023) *The cancer epitope database and analysis resource (CEDAR).* Nucleic acids research 51, no. D1 (2023): D845-D852. [https://doi.org/10.1093/nar/gkac902 ](https://doi.org/10.1093/nar/gkac902)